In [3]:
import os
import sys
import glob
import pickle
import shutil
import zipfile

import numpy as np
import matplotlib.pyplot as plt

### FILES ###
import shutil # delate directory
import zipfile # extract a directory

In [4]:
# DELETE A DIRECTORY FROM CONTENT

folder = "/content/Dataset"

if os.path.exists(folder):
    shutil.rmtree(folder)
    print("Delated directory")
else:
    print("No delated directory")

Delated directory


In [5]:
# EXTRACT ZIP DIRECTORIES

from google.colab import drive
drive.mount('/content/drive')

LOCAL_ROOT = "/content/Dataset"

def unzip_file(zip_path):

    os.makedirs(LOCAL_ROOT, exist_ok=True)

    zip_name = os.path.basename(zip_path)

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(LOCAL_ROOT)

    print(f"{zip_name} has extracted.")

unzip_file("/content/drive/MyDrive/DATASET_SHARP/Python_code_old.zip")
unzip_file("/content/drive/MyDrive/DATASET_SHARP/doppler_traces.zip")
unzip_file("/content/drive/MyDrive/DATASET_SHARP/doppler_traces_S4_S5.zip")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Python_code_old.zip has extracted.
doppler_traces.zip has extracted.
doppler_traces_S4_S5.zip has extracted.


In [6]:
# RETURN THE ACTIONS FOR EACH DIRECTORY

def getActions(folder_path):

    folder_name = os.path.basename(os.path.normpath(folder_path))

    actions = []

    for filename in os.listdir(folder_path):

        prefix = f"{folder_name}_"
        marker = "_stream_"

        if not filename.startswith(prefix):
            continue

        if marker not in filename:
            continue

        remaining = filename[len(prefix):]

        action = remaining.split(marker)[0]

        if action not in actions:
            actions.append(action)

    actions.sort()

    print("Actions:", actions)

    return actions

In [7]:
# LISTA DELLE CARTELLE DEL DATASET

ROOT_PATH = "/content/Dataset/doppler_traces"

folders = []

for folder_name in sorted(os.listdir(ROOT_PATH)):

    folder_path = os.path.join(ROOT_PATH, folder_name)

    # Considera solo le cartelle
    if not os.path.isdir(folder_path):
        continue

    folders.append(folder_path)

print(folders)


['/content/Dataset/doppler_traces/S1a', '/content/Dataset/doppler_traces/S1b', '/content/Dataset/doppler_traces/S1c', '/content/Dataset/doppler_traces/S2a', '/content/Dataset/doppler_traces/S2b', '/content/Dataset/doppler_traces/S3a', '/content/Dataset/doppler_traces/S4a', '/content/Dataset/doppler_traces/S4b', '/content/Dataset/doppler_traces/S5a', '/content/Dataset/doppler_traces/S6a', '/content/Dataset/doppler_traces/S6b', '/content/Dataset/doppler_traces/S7a']


In [8]:
# EXTRACT DATASET

complete_dataset = {}

for i in range(0, len(folders)):

  folder = folders[i]
  print(folder)
  actions = getActions(folder)
  dataset_array = {}

  for action in actions:
    dataset_array[action] = []

  print("Dataset array: ", dataset_array)

  folder_name = os.path.basename(folder)
  print("Folder name: ", folder_name)

  for action in actions:

    for filename in os.listdir(folder):
      if filename.startswith(f"{folder_name}_{action}"):

        file_path = os.path.join(folder, filename)

        with open(file_path, "rb") as fp:
          doppler = pickle.load(fp)

        dataset_array[action].append(doppler)

  complete_dataset[folder_name] = dataset_array

  # ** PRINT **
  print("\n==========================")
  print("Cartella:", folder_name)

  for action in dataset_array:

    print(f"\nAzione: {action}")
    print("Numero stream:", len(dataset_array[action]))

    for i, stream in enumerate(dataset_array[action]):
        print(f"  Stream {i}: shape {np.array(stream).shape}")

/content/Dataset/doppler_traces/S1a
Actions: ['C', 'E', 'H', 'J1', 'J2', 'L', 'R', 'S', 'W']
Dataset array:  {'C': [], 'E': [], 'H': [], 'J1': [], 'J2': [], 'L': [], 'R': [], 'S': [], 'W': []}
Folder name:  S1a

Cartella: S1a

Azione: C
Numero stream: 4
  Stream 0: shape (18766, 100)
  Stream 1: shape (18766, 100)
  Stream 2: shape (18766, 100)
  Stream 3: shape (18766, 100)

Azione: E
Numero stream: 4
  Stream 0: shape (18700, 100)
  Stream 1: shape (18700, 100)
  Stream 2: shape (18700, 100)
  Stream 3: shape (18700, 100)

Azione: H
Numero stream: 4
  Stream 0: shape (19064, 100)
  Stream 1: shape (19064, 100)
  Stream 2: shape (19064, 100)
  Stream 3: shape (19064, 100)

Azione: J1
Numero stream: 4
  Stream 0: shape (8700, 100)
  Stream 1: shape (8700, 100)
  Stream 2: shape (8700, 100)
  Stream 3: shape (8700, 100)

Azione: J2
Numero stream: 4
  Stream 0: shape (8708, 100)
  Stream 1: shape (8708, 100)
  Stream 2: shape (8708, 100)
  Stream 3: shape (8708, 100)

Azione: L
Numero st

In [9]:
# WINDOW OF 1@(340 x 100)
def create_sliding_windows(complete_dataset, window_length=340, stride=340):

  complete_dataset_windows = {}

  for folder_name in complete_dataset:

    dataset = complete_dataset[folder_name]
    all_windows = {}

    for action in dataset:

      data = np.asarray(dataset[action])
      windows_activity = []
      # elements of each action
      num_streams, time_length, num_features = np.array(data).shape
      print(f"Action {action} -> Shape of data: {num_streams}, {time_length}, {num_features}")

      for stream in range(0, num_streams):

        windows_stream = []
        window = []
        # se lo stream è più corto in partenza della grandezza della finestra
        if time_length < window_length:
          window.append(data[stream])
          print("The dataset is less than window size")
          continue

        start = 0
        end = 340
        while end <= time_length:
          window = data[stream][start:end,:]
          windows_stream.append(window)
          start += stride
          end += stride

        #if start < time_length and end > time_length:
          #last_window = np.array(data[stream][start:time_length,:])
          #print("Zero da aggiungere: ", window_length - last_window.shape[0])
          #zero_padding = np.zeros((window_length - last_window.shape[0], last_window.shape[1]), dtype=last_window.dtype)
          #last_window = np.vstack((last_window, zero_padding))
          #windows_stream.append(last_window)
          #print("Shape ultima finestra: ", last_window.shape)
          #print("NUmbers of zeros: ", len(zero_padding))
          #print("last window: ", last_window)

        windows_stream = np.asarray(windows_stream)

        print(f"Activity: {action}")
        print(f"Stream {stream}:")
        print(f"Numero finestre: {len(windows_stream)}")
        print("Shape prima finestra:", windows_stream[0].shape)
        print("Shape ultima finestra:", windows_stream[-1].shape)
        print("Shape completa dello stream:", windows_stream.shape)
        print("---------------------------------------")

        windows_activity.append(np.asarray(windows_stream))

      windows_activity = np.asarray(windows_activity)
      all_windows[action] = windows_activity

    complete_dataset_windows[folder_name] = all_windows
    #print(f"Shape complessiva azione {action}:", windows_dir[action].shape)

  return complete_dataset_windows

wind = create_sliding_windows(complete_dataset)

print("\n==========================")
print("DATASET FINALE")
print("==========================")

for folder_name in wind:

    print(f"\nCartella: {folder_name}")

    for action in wind[folder_name]:

        print(
            f"  {action}: {wind[folder_name][action].shape}"
        )


Action C -> Shape of data: 4, 18766, 100
Activity: C
Stream 0:
Numero finestre: 55
Shape prima finestra: (340, 100)
Shape ultima finestra: (340, 100)
Shape completa dello stream: (55, 340, 100)
---------------------------------------
Activity: C
Stream 1:
Numero finestre: 55
Shape prima finestra: (340, 100)
Shape ultima finestra: (340, 100)
Shape completa dello stream: (55, 340, 100)
---------------------------------------
Activity: C
Stream 2:
Numero finestre: 55
Shape prima finestra: (340, 100)
Shape ultima finestra: (340, 100)
Shape completa dello stream: (55, 340, 100)
---------------------------------------
Activity: C
Stream 3:
Numero finestre: 55
Shape prima finestra: (340, 100)
Shape ultima finestra: (340, 100)
Shape completa dello stream: (55, 340, 100)
---------------------------------------
Action E -> Shape of data: 4, 18700, 100
Activity: E
Stream 0:
Numero finestre: 55
Shape prima finestra: (340, 100)
Shape ultima finestra: (340, 100)
Shape completa dello stream: (55, 340